![Piksel Sandbox](../../assets/piksel_header.png)

# Basic Analysis

## A. Spectral indices and trends

A spectral index combines two or more bands into one value per pixel that highlights a target surface property.
Vegetation indices contrast a band that vegetation reflects strongly (near-infrared) against a band that vegetation absorbs (red).
Water indices contrast a band that water reflects (green) against one that water absorbs (near-infrared).

Annual GeoMAD composites support multi-year analysis cleanly.
Each yearly slice is already a cloud-free, denoised summary of that year's observations, so several years can be compared directly without further preprocessing.

This notebook computes NDVI and NDWI for the Lake Toba area, displays each, and compares NDVI across all years available for the AOI.

## B. Outline

1. Load the Lake Toba area for a multi-year range.
2. Compute and plot NDVI (Normalised Difference Vegetation Index).
3. Compute and plot NDWI (Normalised Difference Water Index).
4. Compare NDVI across all loaded years.

## C. Loading a multi-year dataset

The load call reuses the area and grid from notebooks 03 to 05, widened to a four-year range (2022 to 2025) so that the trend in section F has enough slices to show change.
The annual GeoMAD product currently covers 2022 onward for this AOI.

In [ ]:
import datacube
import matplotlib.pyplot as plt

dc = datacube.Datacube(app="basic_analysis")

query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": ("2022", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

ds = dc.load(**query)
ds

## D. NDVI: vegetation index

The **Normalised Difference Vegetation Index (NDVI)** measures vegetation density.
It contrasts near-infrared reflectance (high over vegetation) against red reflectance (low over vegetation):

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

NDVI ranges from -1 to +1.
Dense vegetation sits near +0.8, bare ground around 0, water below 0.

In [ ]:
nir = ds.nir.astype("float32")
red = ds.red.astype("float32")

ds["ndvi"] = (nir - red) / (nir + red)

ds.ndvi.isel(time=0).plot(cmap="RdYlGn", vmin=-1, vmax=1)

The plot shows the 2022 NDVI for the Lake Toba area.
Vegetation around the lake appears green; the lake itself appears red as the index turns negative; bare or built-up patches sit near zero in yellow.

The line `ds["ndvi"] = ...` attaches the new DataArray to the Dataset as a fifth data variable.
After this assignment `ds.ndvi` carries the same `time`, `y`, and `x` dimensions as the original bands, ready for further selection or plotting.

## E. NDWI: water index

The **Normalised Difference Water Index (NDWI)** highlights open water.
It contrasts green reflectance (relatively high over water) against near-infrared reflectance (very low over water):

$$\text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR}}$$

NDWI is positive over open water and negative over vegetation and bare ground.

In [ ]:
green = ds.green.astype("float32")
nir   = ds.nir.astype("float32")

ds["ndwi"] = (green - nir) / (green + nir)

ds.ndwi.isel(time=0).plot(cmap="RdBu", vmin=-1, vmax=1)

The plot shows Lake Toba in deep blue, where NDWI is strongly positive.
Land surfaces appear red as the index turns negative.

Both NDVI and NDWI now sit in `ds` alongside the original four bands.
Any combination of them can be selected, sliced, or plotted with the same xarray patterns introduced in notebooks 04 and 05.

## F. Multi-year NDVI trend

`ds.ndvi` carries one time slice per year of the loaded range.
The faceting introduced in notebook 05 displays every year side by side on a shared colour scale, so year-over-year vegetation change is visible directly.

In [ ]:
ds.ndvi.plot(col="time", col_wrap=2, cmap="RdYlGn", vmin=-1, vmax=1)

The four panels show NDVI from 2022 to 2025 in a 2 by 2 grid, with the colour scale shared across all panels.
A drop in green intensity between years indicates vegetation loss; a rise indicates regrowth or expansion.
The lake outline stays red across every panel, confirming persistent open water.

For a focused look at part of the scene, `.sel` or `.isel` can narrow the DataArray before plotting; the rest of the pattern stays the same.

## G. Next steps

This notebook closes the beginner spine.
A reader who has worked through 01 to 06 can now connect to the datacube, query a GeoMAD product over a chosen area and time range, work with the returned `xarray.Dataset`, compute spectral indices, and visualise a multi-year trend.

Routes onward:

- The exercise notebook accompanying the spine applies the same patterns to a different area.
- Case studies under `notebooks/English/Case_Studies/` use these patterns on specific topics.
- The advanced track covers cloud masking, raw scenes, and Dask for larger areas.